In [ ]:
!pip install -U accelerate peft transformers
!pip install -q Pillow
!pip install -q pandas scikit-learn tqdm python-dotenv
!pip install torch torchvision --break-system-packages -q
!pip install -q clearml

: 

In [ ]:
import os
import torch

# cuda > mps > cpu, or force with env MAC_DEVICE=cpu|mps|cuda
_force = os.environ.get("MAC_DEVICE", "").strip().lower()
if _force in ("cpu", "mps", "cuda"):
    MAC_DEVICE = torch.device(_force)
    if _force == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("MAC_DEVICE=cuda but CUDA is not available")
    if _force == "mps" and not torch.backends.mps.is_available():
        raise RuntimeError("MAC_DEVICE=mps but MPS is not available")
    MAC_KIND = f"forced {_force}"
elif torch.cuda.is_available():
    MAC_DEVICE = torch.device("cuda")
    MAC_KIND = f"CUDA ({torch.cuda.get_device_name(0)})"
elif torch.backends.mps.is_available():
    MAC_DEVICE = torch.device("mps")
    MAC_KIND = "MPS (Apple GPU)"
else:
    MAC_DEVICE = torch.device("cpu")
    MAC_KIND = "CPU"

print("Training device:", MAC_KIND, "->", MAC_DEVICE)

In [ ]:
import os
import dotenv
dotenv.load_dotenv()

from clearml import Task, Dataset

import pandas as pd
import numpy as np

In [ ]:

Task.set_credentials(
     api_host="https://api.clear.ml",
     web_host="https://app.clear.ml",
     files_host="https://files.clear.ml",
     key=os.getenv('clear_ml_key'),
     secret=os.getenv('clear_ml_secret')
)

In [ ]:
# Utility functions

def parse_basic(path):
    """"
    Note: The structure of the metadata files is as follows:
    dish_id, total_calories, total_mass, total_fat, total_carb, total_protein, ingr_1_id, ingr_1_name, ingr_1_grams, ingr_1_calories, ingr_1_fat, ingr_1_carb, ingr_1_protein, ingr_2_id, ingr_2_name, ...

    Some dishes have more than 5 ingredients, while some have fewer.
    """""
    rows = []

    with open(path) as f:
        for line in f:
            parts = line.strip().split(",")

            try:
                rows.append({
                    "dish_id": parts[0].replace("dish_", ""),
                    "total_calories": float(parts[1]),
                    "total_mass": float(parts[2]),
                    "total_fat": float(parts[3]),
                    "total_carb": float(parts[4]),
                    "total_protein": float(parts[5]),
                })
            except (IndexError, ValueError):
                continue

    return pd.DataFrame(rows)

In [ ]:
from pathlib import Path

# Use ClearML like the main notebook, OR set NUTRITION5K_LOCAL_DIR to a local folder (CSVs + dish_*_rgb.png)
_local = os.environ.get("NUTRITION5K_LOCAL_DIR", "").strip()
if _local:
    local_path = str(Path(_local).resolve())
    print("Dataset path (local):", local_path)
else:
    ds = Dataset.get(
        dataset_name="nutrition5k_dataset",
        dataset_project="NutritionAnalyser"
    )
    local_path = ds.get_local_copy()
    print("Dataset path (ClearML):", local_path)

df1 = parse_basic(os.path.join(local_path, "dish_metadata_cafe1.csv"))
df2 = parse_basic(os.path.join(local_path, "dish_metadata_cafe2.csv"))

df = pd.concat([df1, df2], ignore_index=True)

print(df.shape)
df.head()

#### Map image path


In [ ]:
# add image path column
df["image_path"] = df["dish_id"].apply(
    lambda x: os.path.join(local_path, f"dish_{x}_rgb.png")
)

# filter only rows where image exists
df = df[df["image_path"].apply(os.path.exists)]
print("Final samples:", len(df))

### Feature engineering


In [ ]:
# remove rows with zero calories (these are likely errors in the data and will cause issues with log-transforming the target variable)
df_nozero = df[df["total_calories"] > 0].copy()

# log-transform nutritional values to reduce skewness in the distribution and make it easier for the model to learn
df_nozero["calories_log"] = np.log1p(df_nozero["total_calories"])
df_nozero["protein_log"] = np.log1p(df_nozero["total_protein"])
df_nozero["carb_log"] = np.log1p(df_nozero["total_carb"])
df_nozero["fat_log"] = np.log1p(df_nozero["total_fat"])

In [ ]:
from sklearn.model_selection import train_test_split

df_nozero["cal_bin"] = pd.qcut(df_nozero["total_calories"], q=5, labels=False)

train_df, temp_df = train_test_split(
    df_nozero,
    test_size=0.4,
    random_state=42,
    stratify=df_nozero["cal_bin"]
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["cal_bin"]
)

train_df = train_df.drop(columns=["cal_bin"]).reset_index(drop=True)
val_df = val_df.drop(columns=["cal_bin"]).reset_index(drop=True)
test_df = test_df.drop(columns=["cal_bin"]).reset_index(drop=True)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

train_df = train_df[[
    "image_path",
    "calories_log",
    "protein_log",
    "carb_log",
    "fat_log"
]]

val_df = val_df[[
    "image_path",
    "calories_log",
    "protein_log",
    "carb_log",
    "fat_log"
]]

test_df = test_df[[
    "image_path",
    "calories_log",
    "protein_log",
    "carb_log",
    "fat_log"
]]

#### VLM Fine-Tuning 


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import json

PROMPT = """You are a food nutrition analyzer. Analyze the food image and return ONLY a JSON object with this exact schema:

{
  "nutritional_summary": {
    "calories_kcal": <number>,
    "protein_g": <number>,
    "carbohydrate_g": <number>,
    "fat_g": <number>
  }
}

Rules:
- Return JSON only. No explanation, no markdown, no code blocks.
- All values must be numbers, not strings.
- Do not add extra fields."""

class FoodVLMDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        image.thumbnail((512, 512), Image.Resampling.LANCZOS)

        import math

        target = {
            "nutritional_summary": {
                "calories_kcal": round(math.expm1(row["calories_log"])),
                "protein_g": round(math.expm1(row["protein_log"]), 2),
                "carbohydrate_g": round(math.expm1(row["carb_log"]), 2),
                "fat_g": round(math.expm1(row["fat_log"]), 2),
            }
        }

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": PROMPT}
                ]
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": json.dumps(target)}]
            }
        ]
        return {"messages": messages, "image": image}

#### Load Model + LoRA


In [ ]:
import os
from pathlib import Path

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from peft import PeftModel, LoraConfig, get_peft_model
import torch

# Full-precision Qwen3-VL (no bitsandbytes / Unsloth 4-bit). Override with MAC_BASE_MODEL if needed.
BASE_MODEL_NAME = os.environ.get("MAC_BASE_MODEL", "Qwen/Qwen3-VL-2B-Instruct")

# Separate checkpoint dir from the CUDA notebook defaults
CHECKPOINT_DIR = Path(os.environ.get("FINETUNE_CHECKPOINT_DIR", "./food-vlm-finetuned-mac-best"))

_use_fp32 = os.environ.get("MAC_FP32", "").lower() in ("1", "true", "yes")
if _use_fp32 or MAC_DEVICE.type == "cpu":
    MAC_DTYPE = torch.float32
elif MAC_DEVICE.type == "cuda":
    MAC_DTYPE = torch.float16
else:
    MAC_DTYPE = torch.bfloat16

model = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=MAC_DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model = model.to(MAC_DEVICE)

processor = AutoProcessor.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
print("Processor image size:", processor.image_processor.size)
print("Model dtype:", MAC_DTYPE)

model = PeftModel.from_pretrained(model, "Ateeqq/food-analysis", is_trainable=False)
model = model.merge_and_unload()

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing: on")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

_adapter_files_ok = (
    CHECKPOINT_DIR.is_dir()
    and (CHECKPOINT_DIR / "adapter_config.json").exists()
    and (
        (CHECKPOINT_DIR / "adapter_model.safetensors").exists()
        or (CHECKPOINT_DIR / "adapter_model.bin").exists()
    )
)
if _adapter_files_ok:
    print(f"Resuming trainable LoRA from {CHECKPOINT_DIR.resolve()}")
    model = PeftModel.from_pretrained(model, str(CHECKPOINT_DIR), is_trainable=True)
    ADAPTER_RESUMED = True
else:
    print("Initializing a new LoRA adapter (no prior checkpoint in CHECKPOINT_DIR)")
    model = get_peft_model(model, lora_config)
    ADAPTER_RESUMED = False

model.print_trainable_parameters()

#### Collator + Training Loop


In [ ]:
from torch.utils.data import DataLoader
import torch

# Shorter default on Mac; raise via env MAC_MAX_SEQ_LEN (e.g. 2048) if memory allows
_MAX_SEQ_LEN = int(os.environ.get("MAC_MAX_SEQ_LEN", "1024"))
_BATCH_SIZE = int(os.environ.get("MAC_BATCH_SIZE", "1"))
_NUM_WORKERS = int(os.environ.get("MAC_NUM_WORKERS", "0"))


def collate_fn(batch):
    """Batch samples for the vision-language model and build masked labels for SFT.

    Each sample is expected to provide ``messages`` (chat turns) and ``image`` as
    produced by ``FoodVLMDataset``. The processor tokenizes the full conversation
    (user image + prompt + assistant JSON) and a **user-only** prefix with
    ``add_generation_prompt=True`` so its token IDs align with the start of the
    full sequence.

    Labels are ``input_ids`` with ``-100`` on positions that must not contribute to
    loss: the user/prefix span (including image tokens), padding, and optionally a
    fallback to the full sequence for a row if masking would leave no supervised
    tokens (e.g. aggressive truncation).

    Args:
        batch: List of dicts with keys ``messages`` and ``image``.

    Returns:
        A dict of tensors suitable for ``model(**batch)``, including ``labels``.
    """
    images = [item["image"] for item in batch]
    texts = []
    prefix_texts = []
    for item in batch:
        texts.append(
            processor.apply_chat_template(
                item["messages"], tokenize=False, add_generation_prompt=False
            )
        )
        user_only = item["messages"][:1]
        prefix_texts.append(
            processor.apply_chat_template(
                user_only, tokenize=False, add_generation_prompt=True
            )
        )

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=_MAX_SEQ_LEN,
    )

    prefix_inputs = processor(
        text=prefix_texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=_MAX_SEQ_LEN,
    )

    labels = inputs["input_ids"].clone()
    bsz = labels.size(0)
    for i in range(bsz):
        plen = int(prefix_inputs["attention_mask"][i].sum().item())
        flen = int(inputs["attention_mask"][i].sum().item())
        p_ids = prefix_inputs["input_ids"][i, :plen]
        f_ids = inputs["input_ids"][i, :flen]
        if plen > 0 and flen >= plen and torch.equal(p_ids, f_ids[:plen]):
            assistant_start = plen
        else:
            assistant_start = 0
            for j in range(min(plen, flen)):
                if prefix_inputs["input_ids"][i, j] != inputs["input_ids"][i, j]:
                    assistant_start = j
                    break
            else:
                assistant_start = min(plen, flen)
        labels[i, :assistant_start] = -100

    labels[inputs["attention_mask"] == 0] = -100
    for i in range(bsz):
        if (labels[i] != -100).sum().item() == 0:
            labels[i] = inputs["input_ids"][i].clone()
            labels[i][inputs["attention_mask"][i] == 0] = -100
    inputs["labels"] = labels
    return inputs


train_dataset = FoodVLMDataset(train_df, processor)
train_loader = DataLoader(
    train_dataset, batch_size=_BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=_NUM_WORKERS,
)

val_dataset = FoodVLMDataset(val_df, processor)
val_loader = DataLoader(
    val_dataset, batch_size=_BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=_NUM_WORKERS,
)

test_dataset = FoodVLMDataset(test_df, processor)
test_loader = DataLoader(
    test_dataset, batch_size=_BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=_NUM_WORKERS,
)

In [ ]:
from transformers import get_scheduler
from torch.optim import AdamW
from tqdm import tqdm
import torch


def _model_device(mod):
    for p in mod.parameters():
        return p.device
    return torch.device("cpu")


def _optimizer_state_to_device(opt, dev):
    for st in opt.state.values():
        for k, v in st.items():
            if torch.is_tensor(v):
                st[k] = v.to(dev)


optimizer = AdamW(model.parameters(), lr=1e-4)
num_epochs = 5
device = _model_device(model)

scheduler = get_scheduler(
    "cosine",
    optimizer,
    num_warmup_steps=10,
    num_training_steps=len(train_loader) * num_epochs,
)

training_state_path = CHECKPOINT_DIR / "training_state.pt"
start_epoch = 0
best_val_loss = float("inf")
state = None
if ADAPTER_RESUMED and training_state_path.is_file():
    try:
        state = torch.load(training_state_path, map_location="cpu", weights_only=False)
    except TypeError:
        state = torch.load(training_state_path, map_location="cpu")
    start_epoch = int(state.get("next_epoch", 0))
    best_val_loss = float(state.get("best_val_loss", float("inf")))
    optimizer.load_state_dict(state["optimizer"])
    _optimizer_state_to_device(optimizer, device)
    sched_ok = (
        state.get("num_epochs") == num_epochs
        and state.get("len_train_loader") == len(train_loader)
    )
    if sched_ok:
        scheduler.load_state_dict(state["scheduler"])
    else:
        print(
            "Warning: num_epochs or len(train_loader) changed vs checkpoint; "
            "optimizer restored but scheduler reset to a fresh cosine schedule."
        )
    print(
        f"Loaded training_state.pt — next loop epoch index {start_epoch} "
        f"(run epochs {start_epoch + 1}..{num_epochs} in 1-based logs), best_val_loss={best_val_loss:.4f}"
    )
elif training_state_path.is_file() and not ADAPTER_RESUMED:
    print(
        "Note: training_state.pt exists but no adapter was resumed; starting epoch 0 with a fresh optimizer."
    )

if start_epoch >= num_epochs:
    print(f"Checkpoint says next_epoch={start_epoch} but num_epochs={num_epochs}; nothing to run.")

model.train()
for epoch in range(start_epoch, num_epochs):
    total_loss = 0.0
    n_batches = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item()
        n_batches += 1

    denom = max(n_batches, 1)
    avg_train_loss = total_loss / denom

    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
            outputs = model(**batch)
            if torch.isfinite(outputs.loss):
                val_loss += outputs.loss.item()
                val_batches += 1

    avg_val_loss = val_loss / max(val_batches, 1)
    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        print(f"  → New best model! Saving adapter to {CHECKPOINT_DIR.resolve()} ...")
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(CHECKPOINT_DIR))
        processor.save_pretrained(str(CHECKPOINT_DIR))

    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "next_epoch": epoch + 1,
            "best_val_loss": best_val_loss,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "num_epochs": num_epochs,
            "len_train_loader": len(train_loader),
        },
        training_state_path,
    )

    model.train()